# 1.Import Important Libraries 

In [ ]:
import pandas as pd 
import numpy as np 
import plotly.express as px
import seaborn as sns 
import matplotlib.pyplot as plt 
from plotly.subplots import make_subplots
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

# 2.Load the Datasets

In [ ]:
customers = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv")
orders = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv")
items = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv")
payments = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv")
reviews = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv")
products = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv")
sellers = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv")
geolocation = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv")
translation = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/product_category_name_translation.csv")

In [ ]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": translation
}

# 3.Data Overview

In [ ]:
overview = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()],
    "Missing Values": [df.isnull().sum().sum() for df in datasets.values()],
    "Duplicate Rows": [df.duplicated().sum() for df in datasets.values()]
})

overview

In [ ]:
for name, df in datasets.items():
    print("="*80)
    print(f"{name.upper()}")
    print("="*80)

    print("\nShape:")
    print(df.shape)

    print("\nFirst 10 Rows:")
    display(df.head(10))

    print("\nData Types:")
    display(df.dtypes)

    print("\nMissing Values:")
    display(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

##  Data Overview: Insights and Issues

## 1. Customers Dataset

###  Insights
- Contains **99,441 customers** and **5 columns**.
- No missing values or duplicate records.
- Includes customer ID and location information (ZIP code, city, and state).

###  Issues
- No major issues found.
- `customer_zip_code_prefix` may be converted to a string to preserve leading zeros.

---

## 2. Orders Dataset

###  Insights
- Contains **99,441 orders**.
- Each order has a unique `order_id`.
- Includes timestamps covering the entire order process.

###  Issues
- Missing values in some delivery-related columns:
  - `order_approved_at`: **160**
  - `order_delivered_carrier_date`: **1,783**
  - `order_delivered_customer_date`: **2,965**
- Date columns should be converted from **object** to **datetime**.

---

## 3. Order Items Dataset

###  Insights
- Contains **112,650 order items**.
- Some orders include multiple products.
- No missing values or duplicates.

###  Issues
- `shipping_limit_date` should be converted to **datetime**.

---

## 4. Payments Dataset

###  Insights
- Contains **103,886 payment records**.
- Some orders have more than one payment.
- No missing values or duplicates.

###  Issues
- No major issues found.

---

## 5. Reviews Dataset

###  Insights
- Contains **99,224 reviews**.
- Review scores are complete.
- No duplicate records.

###  Issues
- Many missing values in:
  - `review_comment_title`
  - `review_comment_message`
- These are optional fields, so missing values are expected.
- Date columns should be converted to **datetime**.

---

## 6. Products Dataset

###  Insights
- Contains **32,951 products**.
- Includes product category, dimensions, weight, and photos.
- No duplicate records.

###  Issues
- Missing values exist in product category, description, photos, and dimensions.
- Numeric columns with missing values are stored as **float**.

---

## 7. Sellers Dataset

###  Insights
- Contains **3,095 sellers**.
- No missing values or duplicates.

###  Issues
- No major issues found.

---

## 8. Geolocation Dataset

###  Insights
- Largest dataset with **1,000,163 records**.
- Includes ZIP code, city, state, and geographic coordinates.
- No missing values.

###  Issues
- Contains **261,831 duplicate rows**, which should be checked before analysis.

---

## 9. Category Translation Dataset

###  Insights
- Contains **71 product category translations**.
- No missing values or duplicates.

###  Issues
- No issues found.

---

# Overall Observations

###  Insights
- The dataset contains **9 related tables**.
- The **Orders** table is the main table linking customers, products, sellers, payments, and reviews.
- Most tables are clean with few data quality issues.
- Timestamp columns allow time-based analysis.

###  Data Cleaning Needed
- Convert all date columns to **datetime**.
- Handle missing values in the **Orders** and **Products** tables.
- Keep missing review comments as they are optional.
- Remove duplicate rows from the **Geolocation** table.
- Check relationships between tables after merging.

# Number of Unique Values

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.nunique())

# Insights from Unique Values

### Customers
- Each customer record is unique (`customer_id`).
- `customer_unique_id` has fewer unique values, indicating some customers placed multiple orders.
- Customers are distributed across **4,119 cities** and **27 states**.

### Orders
- Each order is unique.
- The dataset contains **8 order statuses**.
- Purchase timestamps are highly unique, while estimated delivery dates are limited to **459** unique dates.

### Order Items
- Some orders contain multiple products.
- The marketplace includes **32,951 products** sold by **3,095 sellers**.
- Prices and shipping costs have repeated values.

### Payments
- Nearly every order has a payment record.
- Only **5 payment methods** are available.
- Customers can pay in up to **24 installments**.

### Reviews
- Reviews use a **5-star rating** system.
- Review messages are much more diverse than review titles.

### Products
- Contains **32,951 products** across **73 categories**.
- Product descriptions and dimensions vary considerably.

### Sellers
- The marketplace has **3,095 sellers** located in **611 cities** across **23 states**.

### Geolocation
- Covers **19,015 ZIP code prefixes**, **8,011 cities**, and all **27 Brazilian states**.
- Latitude and longitude values are highly diverse, supporting geographical analysis.

### Category Translation
- Contains **71 product categories** with one-to-one Portuguese-to-English translations.

# Missing Values Percentage

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")

    missing = pd.DataFrame({
        "Missing": df.isnull().sum(),
        "Percentage": (df.isnull().mean()*100).round(2)
    })

    display(missing[missing["Missing"] > 0].sort_values("Percentage", ascending=False))

### Insight

- Most datasets are complete with little or no missing data.
- Missing review comments are expected and will be removed.
- Minor missing values in the **Orders** and **Products** tables will be handled during preprocessing.

# Numerical Summary

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.describe())

# Categorical Summary

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.describe(include="object"))

# Distribution of Numerical Features

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

tables = [
    (
        "Order Items",
        items,
        ["price", "freight_value"]
    ),
    (
        "Payments",
        payments,
        ["payment_value", "payment_installments"]
    ),
    (
        "Products",
        products,
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    )
]

for table_name, df, columns in tables:

    fig = make_subplots(
        rows=1,
        cols=len(columns),
        subplot_titles=columns
    )

    for i, col in enumerate(columns, start=1):

        fig.add_trace(
            go.Histogram(
                x=df[col],
                nbinsx=40,
                name=col
            ),
            row=1,
            col=i
        )

    fig.update_layout(
        title=f"{table_name} - Distribution of Numerical Features",
        template="plotly_white",
        showlegend=False,
        height=450,
        width=350 * len(columns),
        bargap=0.05
    )

    fig.show()

# 4.Data Cleaning & Formatting 

## 1. Drop Unnecessary Columns

The review title and review message columns contain free-text data and have a high percentage of missing values. Since this analysis focuses on business insights rather than text analysis, these columns are removed.

In [ ]:
reviews.drop(
    columns=["review_comment_title", "review_comment_message"],
    inplace=True
)

reviews.head()

## 2. Check Duplicate Records

Duplicate rows were checked in all datasets. Only the Geolocation dataset contained duplicate records, which were removed.

In [ ]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicates")

In [ ]:
geolocation.drop_duplicates(inplace=True)

print("Remaining duplicates:", geolocation.duplicated().sum())

## 3. Fix Data Formatting

Date columns were converted from `object` to `datetime` to enable time-based analysis.

In [ ]:
# Orders
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[order_date_cols] = orders[order_date_cols].apply(pd.to_datetime)

# Order Items
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])

# Reviews
review_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

reviews[review_date_cols] = reviews[review_date_cols].apply(pd.to_datetime)

In [ ]:
datetime_columns = {
    "Orders": order_date_cols,
    "Order Items": ["shipping_limit_date"],
    "Reviews": review_date_cols
}

datasets_dict = {
    "Orders": orders,
    "Order Items": items,
    "Reviews": reviews
}

for table, cols in datetime_columns.items():
    print(f"\n{table}")
    display(datasets_dict[table][cols].dtypes)

## 4. Handling Missing Values in Order Date Columns

In [ ]:
# =====================================================
# Fill Missing Date Values Using Business Logic
# =====================================================

# -----------------------------------
# 1. order_approved_at
# -----------------------------------

# Calculate the median approval delay
approval_delay = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dropna().median()

# Fill only orders that should have been approved
mask = (
    orders["order_approved_at"].isna()
    &
    orders["order_status"].isin([
        "approved",
        "processing",
        "invoiced",
        "shipped",
        "delivered"
    ])
)

orders.loc[mask, "order_approved_at"] = (
    orders.loc[mask, "order_purchase_timestamp"] +
    approval_delay
)


# -----------------------------------
# 2. order_delivered_carrier_date
# -----------------------------------

# Calculate the median shipping delay
carrier_delay = (
    orders["order_delivered_carrier_date"] -
    orders["order_approved_at"]
).dropna().median()

# Fill only delivered orders
mask = (
    orders["order_delivered_carrier_date"].isna()
    &
    (orders["order_status"] == "delivered")
)

orders.loc[mask, "order_delivered_carrier_date"] = (
    orders.loc[mask, "order_approved_at"] +
    carrier_delay
)


# -----------------------------------
# 3. order_delivered_customer_date
# -----------------------------------

# Calculate the median delivery delay
delivery_delay = (
    orders["order_delivered_customer_date"] -
    orders["order_delivered_carrier_date"]
).dropna().median()

# Fill only delivered orders
mask = (
    orders["order_delivered_customer_date"].isna()
    &
    (orders["order_status"] == "delivered")
)

orders.loc[mask, "order_delivered_customer_date"] = (
    orders.loc[mask, "order_delivered_carrier_date"] +
    delivery_delay
)


# -----------------------------------
# 4. Check Remaining Missing Values
# -----------------------------------

print("Remaining Missing Values:")
print(
    orders[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ].isna().sum()
)

### Steps Performed

- Calculated the median time difference between:
  - `order_purchase_timestamp` and `order_approved_at`
  - `order_approved_at` and `order_delivered_carrier_date`
  - `order_delivered_carrier_date` and `order_delivered_customer_date`
- Used these median delays to estimate missing dates only for orders whose status indicated that the corresponding event should have occurred.
- Left missing values unchanged for orders with statuses such as **Canceled**, **Unavailable**, or **Created**, since missing dates in these cases represent valid business scenarios rather than missing data.

## 5.Handling Missing Values in the Products Table

In [ ]:
# =====================================================
# Handle Missing Values in Products Table
# =====================================================

# -----------------------------------------------------
# 1. Fill numeric missing values using the median
#    within each product category
# -----------------------------------------------------

numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in numeric_cols:
    products[col] = (
        products
        .groupby("product_category_name")[col]
        .transform(lambda x: x.fillna(x.median()))
    )

# -----------------------------------------------------
# 2. Fill any remaining numeric missing values
#    using the overall median
# -----------------------------------------------------

for col in numeric_cols:
    products[col] = products[col].fillna(products[col].median())

# -----------------------------------------------------
# 3. Verify whether missing product categories
#    can be recovered from existing data
# -----------------------------------------------------

missing_count = products["product_category_name"].isna().sum()

print(f"Missing product categories: {missing_count}")

duplicate_products = products.loc[
    products["product_category_name"].isna(),
    "product_id"
].duplicated().sum()

print(f"Duplicate product_ids among missing categories: {duplicate_products}")

if duplicate_products == 0:
    print(
        "Each missing product belongs to a unique product_id. "
        "Since no other table stores product_category_name, "
        "the missing categories cannot be recovered."
    )

# -----------------------------------------------------
# 4. Fill missing product categories with 'Unknown'
# -----------------------------------------------------

products["product_category_name"] = products[
    "product_category_name"
].fillna("Unknown")

# -----------------------------------------------------
# 5. Verify that all missing values have been handled
# -----------------------------------------------------

print("\nRemaining Missing Values:")
print(products.isnull().sum())

Missing values in the **Orders** table were handled using business logic and the typical order lifecycle observed in the dataset.

### Steps Performed

- Examined the distribution of missing values across all date columns and analyzed the corresponding `order_status` values before performing any imputation.
- Calculated the **median time intervals** between consecutive order events (e.g., purchase to approval, approval to carrier delivery, and carrier delivery to customer delivery) using records with complete information.
- Filled missing date values **only for orders with statuses where the event was expected to occur** by adding the corresponding median time interval to the previous available timestamp.
- Left missing values unchanged for orders with statuses such as **Canceled**, **Unavailable**, and **Created**, since these orders did not progress through the complete order fulfillment process. In these cases, the missing dates represent valid business scenarios rather than missing data.
- Verified that the imputed dates followed the logical chronological order of the order lifecycle and that only appropriate records were modified.

## 6. Detect Outliers

Boxplots were used to identify potential outliers in numerical columns. Outliers were inspected to determine whether they represent valid observations or data errors.

In [ ]:
tables = [
    ("Order Items", items, ["price", "freight_value"]),
    ("Payments", payments, ["payment_value", "payment_installments"]),
    ("Products", products, [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ])
]

for table_name, df, cols in tables:

    fig = make_subplots(
        rows=1,
        cols=len(cols),
        subplot_titles=cols
    )

    for i, col in enumerate(cols, start=1):
        fig.add_trace(
            go.Box(
                y=df[col],
                name=col,
                boxpoints="outliers"
            ),
            row=1,
            col=i
        )

    fig.update_layout(
        title=f"{table_name} - Boxplots",
        template="plotly_white",
        showlegend=False,
        width=350 * len(cols),
        height=500
    )

    fig.show()

In [ ]:
tables = [
    (
        "Order Items",
        items,
        ["price", "freight_value"]
    ),
    (
        "Payments",
        payments,
        ["payment_value", "payment_installments"]
    ),
    (
        "Products",
        products,
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    )
]

outliers = []

for table_name, df, columns in tables:

    for col in columns:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        count = ((df[col] < lower) | (df[col] > upper)).sum()

        outliers.append({
            "Dataset": table_name,
            "Column": col,
            "Outliers": count,
            "Percentage (%)": round((count / len(df)) * 100, 2)
        })

outliers_df = pd.DataFrame(outliers)

display(outliers_df)


In [ ]:
tables = [
    (
        "Order Items",
        items,
        ["price", "freight_value"]
    ),
    (
        "Payments",
        payments,
        ["payment_value", "payment_installments"]
    ),
    (
        "Products",
        products,
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    )
]

outliers = []

for table_name, df, columns in tables:

    print(f"\n{'='*70}")
    print(f"{table_name}")
    print(f"{'='*70}")

    for col in columns:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        # Extract outliers
        outlier_values = df.loc[(df[col] < lower) | (df[col] > upper), col]

        count = len(outlier_values)

        outliers.append({
            "Dataset": table_name,
            "Column": col,
            "Outliers": count,
            "Percentage (%)": round((count / len(df)) * 100, 2)
        })

        print(f"\nColumn: {col}")
        print(f"Outliers: {count}")
        display(outlier_values.describe())

outliers_df = pd.DataFrame(outliers)

## Price Outlier Consistency Check

## 1.Order Items - Price

In [ ]:
# Detect price outliers using the IQR method
Q1 = items["price"].quantile(0.25)
Q3 = items["price"].quantile(0.75)
IQR = Q3 - Q1

price_outliers = items[
    items["price"] > (Q3 + 1.5 * IQR)
]

# Merge with the Products table
price_products = price_outliers.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Merge with the translation table
price_products = price_products.merge(
    translation,
    on="product_category_name",
    how="left"
)

# Summary by English category names
price_summary = (
    price_products
    .groupby("product_category_name_english")["price"]
    .agg(
        Count="count",
        Minimum_Price="min",
        Median_Price="median",
        Maximum_Price="max"
    )
    .sort_values(by="Count", ascending=False)
)

price_summary

### Insight

The analysis of price outliers across product categories indicates that the majority of high-priced products are concentrated in categories where premium pricing is expected, such as **Health & Beauty**, **Watches & Gifts**, **Sports & Leisure**, **Auto**, and **Cool Stuff**.

These categories naturally include luxury, branded, or high-value products, explaining the presence of higher prices. Additionally, the maximum prices observed (e.g., over 3,000–4,000) are plausible for these product types and do not suggest unrealistic or erroneous values.

Several categories contain only one or a few outlier products. Although these observations are less frequent, their prices remain reasonable for their respective categories and do not indicate obvious data quality issues.

Overall, the detected price outliers represent genuine business variability rather than incorrect or anomalous records.

## 2.Freight Value

In [ ]:
freight = items.merge(
    products[["product_id","product_weight_g"]],
    on="product_id"
)

freight[["freight_value","product_weight_g"]].corr()

plt.figure(figsize=(8,5))

plt.scatter(
    freight["product_weight_g"],
    freight["freight_value"],
    alpha=0.3
)

plt.xlabel("Product Weight")
plt.ylabel("Freight Value")
plt.title("Weight vs Freight")

plt.show()


In [ ]:
correlation = freight["product_weight_g"].corr(freight["freight_value"])

print(f"Correlation: {correlation:.3f}")

### Insight

The analysis shows a **moderately strong positive correlation** between product weight and freight cost (**r = 0.609**), indicating that heavier products generally incur higher shipping costs. Although other factors such as shipping distance and carrier policies also influence freight charges, the observed outliers follow expected business patterns and are therefore considered **consistent** rather than data quality issues.

## 3. Payment Value

In [ ]:
order_summary = items.groupby("order_id").agg(
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
)

payment_summary = payments.groupby("order_id").agg(
    total_payment=("payment_value", "sum")
)

comparison = (
    order_summary
    .join(payment_summary)
)

comparison["expected_total"] = (
    comparison["total_price"] +
    comparison["total_freight"]
)

comparison["difference"] = (
    comparison["total_payment"] -
    comparison["expected_total"]
)

comparison.sort_values(
    "difference",
    ascending=False
).head(20)

### Insight

After comparing the total payment amount with the expected order value (product price + freight cost), the differences between the two decreased substantially. This indicates that most of the initially detected payment outliers were explained by shipping costs rather than anomalies.

The remaining differences are relatively small compared to the total order values and are likely attributable to factors such as payment processing, installment payments, or minor financial adjustments. No evidence of unrealistic or erroneous payment values was observed.

Overall, the payment value outliers are considered **consistent** with normal business transactions.

## 4.Payment Installments

In [ ]:
payments["payment_installments"].value_counts().sort_index()

In [ ]:
payments.groupby(
    "payment_installments"
)["payment_value"].mean()

### Insight

After comparing payment value with the number of payment installments, the values generally increase as the installment count increases. This indicates that customers tend to use installment plans for higher-value purchases.

Some fluctuations were observed in certain installment categories, but they are likely caused by differences in transaction volume or the presence of high-value orders. No unrealistic patterns or major inconsistencies were identified.

Overall, the relationship between payment installments and payment value is considered **consistent** with normal customer payment behavior.

## 5.Product Weight

In [ ]:
# Detect heavy products using the IQR method
Q1 = products["product_weight_g"].quantile(0.25)
Q3 = products["product_weight_g"].quantile(0.75)
IQR = Q3 - Q1

heavy_products = products[
    products["product_weight_g"] > (Q3 + 1.5 * IQR)
]

# Merge with the translation table
heavy_products = heavy_products.merge(
    translation,
    on="product_category_name",
    how="left"
)

# Summary by English category names
heavy_summary = (
    heavy_products
    .groupby("product_category_name_english")["product_weight_g"]
    .agg(
        Count="count",
        Median_Weight="median",
        Maximum_Weight="max"
    )
    .sort_values(by="Count", ascending=False)
)

heavy_summary

### Insight

After analyzing the categories containing heavy products, the highest number of heavy items is concentrated in categories such as **furniture_decor**, **housewares**, **bed_bath_table**, and **auto**. These categories naturally contain larger and bulkier products, which explains their higher product weights.

Some categories, such as **furniture_mattress_and_upholstery** and **office_furniture**, show high median weights, indicating that their products are consistently heavier rather than being affected only by extreme values.

Overall, the product weight distribution is considered **consistent** with typical e-commerce product characteristics, where furniture, home, and large-item categories contribute the majority of heavy products.

## 6.Product Dimensions

In [ ]:
products["volume"] = (
    products["product_length_cm"] *
    products["product_width_cm"] *
    products["product_height_cm"]
)

products[[
    "volume",
    "product_weight_g"
]].corr()

### Insight

After analyzing the correlation between **product volume** and **product weight**, a strong positive relationship was observed with a correlation coefficient of **0.80**.

This indicates that products with larger dimensions tend to have higher weights, which is expected for physical products where size and weight are usually related. However, the correlation is not perfect, meaning some products may have high volume but relatively low weight due to differences in material density.

Overall, the relationship between product volume and weight is considered **consistent** with normal product characteristics.

## 7.Weight vs Dimensions

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(
    products["volume"],
    products["product_weight_g"],
    alpha=.3
)

plt.xlabel("Volume")
plt.ylabel("Weight")

plt.show()

### Insight
The scatter plot shows a positive relationship between product volume and product weight, meaning larger products generally weigh more. Although there is some variation and a few outliers, the relationship is generally consistent with normal product characteristics.

# Download cleaned tables

In [ ]:
customers.to_csv("customers_clean.csv", index=False)
orders.to_csv("orders_clean.csv", index=False)
items.to_csv("order_items_clean.csv", index=False)
payments.to_csv("payments_clean.csv", index=False)
products.to_csv("products_clean.csv", index=False)
reviews.to_csv("reviews_clean.csv", index=False)
sellers.to_csv("sellers_clean.csv", index=False)
geolocation.to_csv("geolocation_clean.csv", index=False)
translation.to_csv("category_translation_clean.csv", index=False)

# 5.Data Integration and Merging

In [ ]:
orders_features = orders.copy()

orders_features.shape

In [ ]:
orders_features = orders_features.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

orders_features.head()

In [ ]:
order_items_features = (
    items
    .groupby("order_id")
    .agg(
        total_items=("product_id", "count"),
        total_sales=("price", "sum"),
        total_freight=("freight_value", "sum"),
        avg_item_price=("price", "mean"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

orders_features = orders_features.merge(
    order_items_features,
    on="order_id",
    how="left"
)

In [ ]:
payment_features = (
    payments
    .groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
        payment_methods=("payment_type", "nunique"),
        main_payment_method=("payment_type", lambda x: x.mode()[0])
    )
    .reset_index()
)
payment_features = (
    payments
    .groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
        payment_methods=("payment_type", "nunique"),
        main_payment_method=("payment_type", lambda x: x.mode()[0])
    )
    .reset_index()
)

In [ ]:
review_features = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

orders_features = orders_features.merge(
    review_features,
    on="order_id",
    how="left"
)

In [ ]:
items_products = (
    items
    .merge(products, on="product_id", how="left")
    .merge(translation, on="product_category_name", how="left")
)

product_features = (
    items_products
    .groupby("order_id")
    .agg(
        product_categories=(
            "product_category_name_english",
            lambda x: ", ".join(sorted(x.dropna().unique()))
        ),
        avg_product_weight=("product_weight_g", "mean"),
        avg_product_photos=("product_photos_qty", "mean")
    )
    .reset_index()
)

orders_features = orders_features.merge(
    product_features,
    on="order_id",
    how="left"
)

## Feature Engineering

In [ ]:
# Total order value
orders_features["total_order_value"] = (
    orders_features["total_sales"] +
    orders_features["total_freight"]
)

# Purchase date features
orders_features["purchase_year"] = (
    orders_features["order_purchase_timestamp"].dt.year
)

orders_features["purchase_month"] = (
    orders_features["order_purchase_timestamp"].dt.month
)

orders_features["purchase_day"] = (
    orders_features["order_purchase_timestamp"].dt.day
)

orders_features["purchase_weekday"] = (
    orders_features["order_purchase_timestamp"].dt.day_name()
)

orders_features["purchase_hour"] = (
    orders_features["order_purchase_timestamp"].dt.hour
)

# Delivery duration
orders_features["delivery_days"] = (
    orders_features["order_delivered_customer_date"] -
    orders_features["order_purchase_timestamp"]
).dt.days

# Delay days
orders_features["delay_days"] = (
    orders_features["order_delivered_customer_date"] -
    orders_features["order_estimated_delivery_date"]
).dt.days

# Late delivery flag
orders_features["is_late"] = (
    orders_features["delay_days"] > 0
).astype(int)

## The Created New Features

To enrich the analytical dataset, several new features are created from the existing data. These engineered features provide additional insights into customer purchasing behavior, order value, and delivery performance, making the dataset more suitable for exploratory data analysis and predictive modeling.

| Feature | Description |
|----------|-------------|
| **total_order_value** | Calculates the total value of an order by adding the product sales and freight cost. |
| **purchase_year** | Extracts the year when the order was placed. |
| **purchase_month** | Extracts the month when the order was placed. |
| **purchase_day** | Extracts the day of the month when the order was placed. |
| **purchase_weekday** | Identifies the day of the week when the order was placed. |
| **purchase_hour** | Extracts the hour of the day when the purchase occurred. |
| **delivery_days** | Calculates the total number of days between purchase and customer delivery. |
| **delay_days** | Measures the difference between the actual and estimated delivery dates. Positive values indicate delayed deliveries, while negative values indicate early deliveries. |
| **is_late** | Indicates whether an order was delivered late (1) or on time/early (0). |

# 6.EDA

# Sales Performance Analysis

This section evaluates the overall sales performance of the business by examining order trends, seasonality, customer purchasing behavior, and regional revenue distribution. These analyses help identify growth opportunities, peak sales periods, and the company's strongest markets.

## 1.How has the number of orders changed over time?

In [ ]:
monthly_orders = (
    orders_features
    .groupby(["purchase_year", "purchase_month"])
    .size()
    .reset_index(name="Number of Orders")
)

monthly_orders["Date"] = pd.to_datetime(
    monthly_orders["purchase_year"].astype(str)
    + "-"
    + monthly_orders["purchase_month"].astype(str)
)

# Remove incomplete months
monthly_orders = monthly_orders[
    monthly_orders["Date"] < "2018-09-01"
]

fig = px.line(
    monthly_orders,
    x="Date",
    y="Number of Orders",
    markers=True,
    title="Monthly Order Trend",
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=7)
)

fig.update_layout(
    template="plotly_white",
    title={
        "text":"Monthly Order Trend",
        "x":0.5,
        "font":dict(size=22)
    },
    xaxis_title="Month",
    yaxis_title="Number of Orders",
    hovermode="x unified",
    height=600
)

fig.show()

### Insight

- Monthly orders increased significantly from 2016 to 2018, indicating continuous business growth.
- The highest number of orders was recorded in **November 2017**, suggesting the impact of seasonal shopping events(Black Friday).
- Despite minor month-to-month fluctuations, order volumes remained consistently high throughout 2018, reflecting stable customer demand and strong marketplace performance.

## 2.Which months show the strongest seasonal sales performance?

In [ ]:
import calendar

monthly_sales = (
    orders_features
    .groupby("purchase_month", as_index=False)["total_order_value"]
    .sum()
)

monthly_sales["Month"] = monthly_sales["purchase_month"].apply(
    lambda x: calendar.month_name[x]
)

month_order = list(calendar.month_name)[1:]

fig = px.bar(
    monthly_sales,
    x="Month",
    y="total_order_value",
    color="total_order_value",
    color_continuous_scale="Blues",
    text_auto=".2s",
    category_orders={"Month": month_order},
    title="Total Revenue by Month"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Month",
    yaxis_title="Total Revenue (R$)",
    coloraxis_showscale=False
)

fig.show()

### Insight

The monthly revenue analysis shows that **May** generated the highest total revenue, followed closely by **July** and **August**, indicating stronger customer spending during the middle of the year.

On the other hand, **September** recorded the lowest revenue, suggesting a temporary decline in sales activity. Despite these fluctuations, revenue remains relatively stable across most months, indicating consistent customer purchasing behavior throughout the year with only moderate seasonal variation.

## 3.During which hours are customers most likely to place orders?

In [ ]:
hourly_orders = (
    orders_features
    .groupby("purchase_hour")
    .size()
    .reset_index(name="Orders")
)

fig = px.line(
    hourly_orders,
    x="purchase_hour",
    y="Orders",
    markers=True,
    title="Orders by Hour of Day"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Hour",
    yaxis_title="Number of Orders"
)

fig.show()

### Insight

- Order activity is lowest during the late-night and early-morning hours.
- Customer purchases increase rapidly after **7:00 AM**, reaching their highest level around **10:00 AM**.
- Demand remains consistently high throughout the afternoon before gradually decreasing in the evening.
- The results indicate that customers are most active during regular business hours, making this the optimal period for promotions and marketing campaigns.

## 4. Which customer states generate the highest revenue?

In [ ]:
state_sales = (
    orders_features
    .groupby("customer_state")["total_order_value"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    state_sales.head(15),
    x="customer_state",
    y="total_order_value",
    color="total_order_value",
    text_auto=".2s",
    title="Top 15 States by Revenue"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="State",
    yaxis_title="Revenue"
)

fig.show()

### Insight

- **SP** is the highest revenue-generating state, contributing approximately **6M** in total sales.
- **RJ** and **MG** are the next top-performing states, although their revenue is considerably lower than SP.
- Revenue declines noticeably after the top five states, indicating that sales are concentrated in a limited number of regions.
- Expanding marketing and sales efforts in lower-performing states could help improve overall revenue distribution.

## 5. Which product categories generate the highest revenue?

In [ ]:

# Merge order items with products and category translation
items_products = (
    items
    .merge(products, on="product_id", how="left")
    .merge(translation, on="product_category_name", how="left")
)

# Calculate revenue by category
category_sales = (
    items_products
    .groupby("product_category_name_english", as_index=False)["price"]
    .sum()
    .sort_values("price", ascending=False)
)

# Keep only the Top 10 categories
top10 = category_sales.head(10)

# Plot
fig = px.bar(
    top10,
    x="price",
    y="product_category_name_english",
    orientation="h",
    text_auto=".2s",
    color="price",
    color_continuous_scale="Blues",
    title="Top 10 Product Categories by Revenue"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Revenue (R$)",
    yaxis_title="Product Category",
    coloraxis_showscale=False,
    height=600
)

fig.update_yaxes(categoryorder="total ascending")

fig.show()

### Insight

- Health & Beauty is the highest revenue-generating product category, contributing approximately R$1.3M in sales.
- Watches & Gifts and Bed, Bath & Table follow closely, with each generating over R$1M in revenue.
- Revenue is relatively well distributed across the top 10 categories, ranging from R$490K to R$1.3M, indicating a diversified product portfolio.
- The strong performance of home, lifestyle, and technology-related categories suggests these segments are the primary drivers of overall revenue.

## Business Overview

In [ ]:

kpis = pd.DataFrame({
    "KPI": [
        "Total Orders",
        "Total Revenue (R$)",
        "Average Order Value (R$)",
        "Unique Customers"
    ],
    "Value": [
        len(orders_features),
        round(orders_features["total_order_value"].sum(), 2),
        round(orders_features["total_order_value"].mean(), 2),
        orders_features["customer_unique_id"].nunique()
    ]
})

kpis

# Logistics & Delivery Analysis

## 1. Which states have the longest average delivery time?

In [ ]:
state_delivery = (
    orders_features
    .groupby("customer_state", as_index=False)["delivery_days"]
    .mean()
    .sort_values("delivery_days", ascending=False)
    .head(15)
)

fig = px.bar(
    state_delivery,
    x="delivery_days",
    y="customer_state",
    orientation="h",
    text_auto=".1f",
    color="delivery_days",
    color_continuous_scale="Reds",
    title="Top 15 States by Average Delivery Time"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Average Delivery Days",
    yaxis_title="State",
    coloraxis_showscale=False
)

# Show the highest value at the top
fig.update_yaxes(categoryorder="total ascending")

fig.show()

### Insight

- RR has the longest average delivery time at approximately 29 days, followed by AP and AM, indicating slower deliveries in northern states.
- Most of the states in the top 15 require between 18 and 29 days for order delivery, showing noticeable regional differences in shipping performance.
- Northern and northeastern states dominate the list, suggesting that greater distances and logistics challenges contribute to longer delivery times.
- Optimizing distribution routes and expanding logistics coverage in these regions could help reduce delivery times and improve customer satisfaction.

## 2. What percentage of orders were delivered late?

In [ ]:
late_orders = (
    orders_features["is_late"]
    .value_counts()
    .rename(index={0: "On Time", 1: "Late"})
    .reset_index()
)

late_orders.columns = ["Delivery Status", "Orders"]

fig = px.pie(
    late_orders,
    names="Delivery Status",
    values="Orders",
    hole=0.55,
    color="Delivery Status",
    title="Late vs On-Time Deliveries"
)

fig.update_traces(textposition="inside", textinfo="percent+label")

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

### Insight

- The vast majority of orders (93.4%) are delivered on time, indicating a highly reliable delivery process.
- Only 6.57% of orders are delivered late, reflecting strong overall logistics performance.
- The low late-delivery rate suggests that estimated delivery dates are generally accurate and operational efficiency is high.
- Reducing the remaining late deliveries could further improve customer satisfaction, particularly in regions with longer average delivery times.

## 3. Which months experience the highest number of late deliveries?

In [ ]:
late_months = (
    orders_features[orders_features["is_late"] == 1]
    .groupby("purchase_month", as_index=False)
    .size()
)

late_months.columns = ["Month", "Late Orders"]

late_months["Month Name"] = late_months["Month"].apply(
    lambda x: calendar.month_name[x]
)

fig = px.bar(
    late_months,
    x="Month Name",
    y="Late Orders",
    text_auto=True,
    color="Late Orders",
    color_continuous_scale="Reds",
    title="Late Deliveries by Month"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Month",
    yaxis_title="Late Orders",
    coloraxis_showscale=False
)

fig.show()

### Insight

- Late deliveries are highest in **March (1,444 orders)**, followed by **February (976 orders)** and **November (904 orders)**.
- The number of late deliveries fluctuates throughout the year, with lower delays observed during the middle months such as **June and July**.
- The increase in late deliveries during certain months may indicate seasonal demand, higher order volumes, or logistics capacity challenges.

## 4. Which customer states have the highest late delivery rates?

In [ ]:
late_rate = (
    orders_features
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "count"),
        late_orders=("is_late", "sum")
    )
    .reset_index()
)

late_rate["Late Rate (%)"] = (
    late_rate["late_orders"] /
    late_rate["total_orders"] * 100
)

late_rate = late_rate.sort_values(
    "Late Rate (%)",
    ascending=False
)

fig = px.bar(
    late_rate,
    x="customer_state",
    y="Late Rate (%)",
    text_auto=".1f",
    color="Late Rate (%)",
    color_continuous_scale="Reds",
    title="Late Delivery Rate by State"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="State",
    yaxis_title="Late Delivery Rate (%)",
    coloraxis_showscale=False
)

fig.show()

### Insight

- The highest late delivery rates are observed in **Alagoas (AL) (20.6%)**, **Maranhão (MA) (16.7%)**, and **Sergipe (SE) (14.6%)**, indicating these states experience the most delivery delays.
- Most states have late delivery rates below **11%**, with several clustered between **8% and 11%**, suggesting relatively consistent delivery performance across the majority of regions.
- The lowest late delivery rates are found in **Amapá (AP) (2.8%)**, **Rondônia (RO) (2.7%)**, and **Amazonas (AM)**, indicating more reliable on-time deliveries in these states.

## Business Overview

In [ ]:
logistics_kpis = pd.DataFrame({
    "KPI": [
        "Average Delivery Time (Days)",
        "Average Early Delivery (Days)",
        "Late Delivery Rate (%)",
        "On-Time Delivery Rate (%)"
    ],
    "Value": [
        round(orders_features["delivery_days"].mean(), 2),
        round(abs(orders_features["delay_days"].mean()), 2),
        round(orders_features["is_late"].mean() * 100, 2),
        round((1 - orders_features["is_late"].mean()) * 100, 2)
    ]
})

logistics_kpis

# 3. Customer Satisfaction Analysis

## 1. How are customer review scores distributed?

In [ ]:
score_dist = (
    reviews["review_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)
score_dist.columns = ["Review Score", "Count"]

fig = px.bar(
    score_dist,
    x="Review Score",
    y="Count",
    text_auto=True,
    color="Review Score",
    color_continuous_scale="RdYlGn",
    title="Distribution of Customer Review Scores"
)

fig.update_layout(
    template="plotly_white",
    title={"text": "Distribution of Customer Review Scores", "x": 0.5, "font": {"size": 22}},
    xaxis_title="Review Score",
    yaxis_title="Number of Reviews",
    coloraxis_showscale=False,
    height=500,
    xaxis={"tickmode": "linear", "dtick": 1}
)

fig.show()


### Insight
- The distribution is **heavily skewed toward positive ratings**: score **5** is by far the most common, accounting for the majority of all reviews.
- Scores **1** and **2** together represent a small but meaningful fraction of responses, indicating a minority of poor experiences.
- The high concentration of 5-star ratings reflects overall customer satisfaction, but monitoring low scores is essential for continuous improvement.


## 2. Does late delivery reduce customer satisfaction?

In [ ]:
avg_review = (
    orders_features.groupby("is_late")["review_score"]
    .mean()
    .reset_index()
)

avg_review["Delivery Status"] = avg_review["is_late"].map({
    0: "On Time",
    1: "Late"
})

fig = px.bar(
    avg_review,
    x="Delivery Status",
    y="review_score",
    color="Delivery Status",
    text=avg_review["review_score"].round(2),
    title="Average Customer Satisfaction by Delivery Status",
    labels={
        "review_score": "Average Review Score",
        "Delivery Status": "Delivery Status"
    },
    template="plotly_white"
)

fig.update_traces(textposition="outside")
fig.update_layout(
    title_x=0.5,
    showlegend=False,
    yaxis=dict(range=[0, 5])
)

fig.show()

### Insight

- Customers whose orders were delivered **on time** gave an average review score of **4.21/5**, indicating a high level of satisfaction.
- In contrast, customers who received **late deliveries** gave an average review score of only **2.27/5**.
- This substantial drop of nearly **2 rating points** suggests that delivery delays have a strong negative impact on customer satisfaction.
- Improving delivery reliability and reducing late shipments could significantly enhance customer experience and overall review ratings.

## 3.Which States Have the Highest Customer Satisfaction?

In [ ]:
state_reviews = (
    orders_features.groupby("customer_state", as_index=False)["review_score"]
    .mean()
    .sort_values("review_score", ascending=False)
)

fig = px.bar(
    state_reviews,
    x="customer_state",
    y="review_score",
    color="review_score",
    title="Average Customer Review Score by State",
    labels={
        "customer_state": "State",
        "review_score": "Average Review Score"
    },
    template="plotly_white"
)

fig.update_layout(
    title_x=0.5,
    coloraxis_showscale=False
)

fig.show()

### Insight

- Customer satisfaction is consistently high across most states, with average review scores generally ranging between **3.5 and 4.3 out of 5**.
- States such as **AM, AP, and PR** have the highest average customer ratings, indicating a consistently positive customer experience.
- While some states, including **RR, MA, and AL**, have relatively lower average review scores, the differences between states are modest.
- Overall, the results suggest that **geographic location has a limited impact on customer satisfaction** compared to operational factors such as delivery performance. Improving service quality in lower-rated states may further enhance the overall customer experience.

## Business Overview

In [ ]:
customer_satisfaction_kpis = pd.DataFrame({
    "KPI": [
        "Average Review Score",
        "5-Star Reviews",
        "Late Deliveries",
        "Customer Satisfaction Rate (%)"
    ],
    "Value": [
        round(orders_features["review_score"].mean(), 2),
        (orders_features["review_score"] == 5).sum(),
        orders_features["is_late"].sum(),
        round((orders_features["review_score"] >= 4).mean() * 100, 2)
    ]
})

customer_satisfaction_kpis

# EDA Summary

- Orders grew steadily from **2016 to 2018**, peaking in **November 2017** due to **Black Friday** demand.
- **May** had the highest monthly revenue; **September** recorded the lowest.
- **São Paulo (SP)** dominates revenue at ~**R$6M**, far ahead of **RJ** and **MG**.
- **Health & Beauty** is the top product category (~R$1.3M), followed by **Watches & Gifts** and **Bed, Bath & Table**.
- **93.4%** of orders are delivered on time; only **6.57%** arrive late.
- **Roraima (RR)** has the longest average delivery time at ~**29 days**, with northern states consistently slowest.
- Late deliveries peak in **March** and **February**; **June and July** see the fewest delays.
- **Alagoas (AL)** has the highest late delivery rate at **20.6%**, followed by **MA (16.7%)** and **SE (14.6%)**.
- Review scores skew heavily toward **5 stars**, reflecting strong overall satisfaction.
- On-time deliveries average **4.21/5** vs. only **2.27/5** for late deliveries — a nearly **2-point drop**.
- **Delivery performance is the strongest driver of customer satisfaction**, outweighing region or order value.

# Business Recommendations

- Scale up **inventory and logistics ahead of November** to handle Black Friday demand without delays.
- Run targeted campaigns in **September, January, and February** to lift sales during low-revenue months.
- Invest in **logistics infrastructure in northern and northeastern states** (RR, AL, MA, SE) to cut late delivery rates and boost satisfaction.
- Double down on top-performing categories — **Health & Beauty, Watches & Gifts, and Bed, Bath & Table** — through promotions and better visibility.
- Treat **on-time delivery as a top priority**: even small improvements will directly raise review scores and customer retention.
- Investigate **1- and 2-star reviews** at the category and seller level to identify and fix recurring pain points.
